# Grupo 7 - Examen Práctico RL

Integrantes: 

- Diego Valenzuela 22309
- Daniel Dubón 22233
- Joaquín Puente 22296
- Christian Echeverria 221441
- Mario Betancurt 23440

In [1]:
import numpy as np
from collections import defaultdict

# ESPECIFICACIÓN DEL MDP
# Estado: (nivel_inventario, dias_hasta_vencimiento, demanda_promedio_7dias)
# nivel_inventario:       [0, 10, 20, ..., 100]              — 10 niveles
# dias_hasta_vencimiento: [1, 7, 14, 30, 60]                 — 5 niveles
# demanda_promedio_7dias: [bajo, medio, alto, crítico]        — 4 niveles
# Total: 200 estados | Acciones: [0, 10, 20, 30, 40, 50] unidades — 6 acciones

def transition(state, action):
    inventory, days_to_expiry, demand_level = state
    demand_map = {'bajo': 5, 'medio': 15, 'alto': 25, 'crítico': 40}
    daily_demand = demand_map[demand_level]
    new_inventory = min(100, max(0, inventory + action - daily_demand))
    new_days = max(1, days_to_expiry - 1)
    new_demand = demand_level
    return (new_inventory, new_days, new_demand)

def reward(state, action, next_state):
    new_inventory, new_days, _ = next_state
    inventory_reward = new_inventory * 0.5
    expiry_penalty = -10 if new_days <= 7 else 0
    order_penalty = -2 if action > 0 else 0
    return inventory_reward + expiry_penalty + order_penalty

def train(env, episodes=1000):
    Q = defaultdict(lambda: np.zeros(6))
    alpha = 0.9
    gamma = 0.99
    epsilon = 0.05
    for episode in range(episodes):
        state = env.reset()
        done = False
        while not done:
            if np.random.random() < epsilon:
                action = np.random.randint(6)
            else:
                action = np.argmax(Q[state])
            next_state, reward, done, _ = env.step(action)
            next_action = np.argmax(Q[next_state])
            td_target = reward + gamma * Q[next_state][next_action]
            Q[state][action] += alpha * (td_target - Q[state][action])
            state = next_state
    return Q

def preprocess_state(state):
    inventory, days_to_expiry, demand_level = state
    demand_map = {'bajo': 0, 'medio': 1, 'alto': 2, 'crítico': 3}
    features = np.array([
        inventory,
        days_to_expiry,
        demand_map[demand_level]
    ])
    return features

class LinearApproximator:
    def __init__(self, n_features=3, n_actions=6):
        self.weights = np.zeros((n_actions, n_features))
    def predict(self, features, action):
        return np.dot(self.weights[action], features)
    def update(self, features, action, target, alpha=0.01):
        prediction = self.predict(features, action)
        error = target - prediction
        self.weights[action] += alpha * error * features

def evaluate_policy(Q, env, episodes=100):
    total_rewards = []
    for episode in range(episodes):
        state = env.reset()
        episode_reward = 0
        done = False
        while not done:
            action = np.argmax(Q[state])
            next_state, reward, done, _ = env.step(action)
            episode_reward += reward
            state = next_state
        total_rewards.append(episode_reward)
    return {
        'mean_reward': np.mean(total_rewards),
        'std_reward': np.std(total_rewards),
        'min_reward': np.min(total_rewards)
    }

# MÉTRICAS DE ENTRENAMIENTO
# Episodio    Recompensa    Error TD    Política dominante
# 100         42.3          8.42        pedir_50
# 500         48.1          7.12        pedir_50
# 1000        51.7          5.21        pedir_50
# Varianza entre episodios: 0.8
# Política greedy: pedir 50 en 94% de estados

# RESULTADOS EN PRODUCCIÓN
resultados_produccion = {
    'stockouts_por_semana': 23,
    'productos_vencidos_por_semana': 41,
    'costo_almacenamiento_semanal': 8400,
    'costo_objetivo_semanal': 3200,
    'satisfaccion_cliente': 0.61
}

# RESULTADOS EN SIMULACIÓN
resultados_simulacion = {
    'mean_reward': 51.2,
    'std_reward': 0.9,
    'min_reward': 48.3
}

### Entregable 7.1 — Validación del gap simulación–producción

Se valida la dirección del análisis previo, pero se precisan las atribuciones y se separa lo demostrado por el código de lo que solo es una hipótesis. El cociente real de costo es $8400/3200 = 2.625$, es decir, aproximadamente **2.6×** el objetivo.


| Síntoma observado | Causa probable y componente | Evidencia en el código |
|---|---|---|
| **Costo de almacenamiento 2.6×** y política `pedir_50` en 94% | **Función de recompensa mal alineada.** Premia mantener inventario y casi no diferencia el tamaño del pedido. | `inventory_reward = new_inventory * 0.5` entrega hasta +50 por paso; `order_penalty = -2 if action > 0 else 0` cobra lo mismo por 10 que por 50 unidades. Por ello sobreabastecer puede maximizar el retorno simulado aunque sea caro en producción. |
| **41 productos vencidos/semana** | **Reward y modelo de transición incompletos.** El costo de vencer es binario y no proporcional a unidades; el estado tampoco representa lotes con edades distintas. | `expiry_penalty = -10 if new_days <= 7 else 0` aplica la misma penalización con una o cien unidades. `new_days = max(1, days_to_expiry - 1)` solo baja un contador y nunca descarta inventario vencido ni reinicia la vida útil de las unidades nuevas. El simulador no puede reproducir adecuadamente las 41 mermas. |
| **23 stockouts/semana** | **Reward sin costo de demanda perdida**, **transición que oculta el faltante** y **tabla Q sin generalización**. | `max(0, inventory + action - daily_demand)` recorta el inventario en cero, pero no conserva `lost_sales` ni penaliza su magnitud. Además, `defaultdict(lambda: np.zeros(6))` deja sin conocimiento los estados no vistos; la política greedy elige allí la acción 0. |
| **Satisfacción 0.61** | **Objetivo de entrenamiento/evaluación distinto al KPI real.** Servicio, stockouts y satisfacción no aparecen en el retorno. | `reward(...)` solo incluye inventario, umbral de vencimiento y un costo fijo por ordenar. `evaluate_policy(...)` reporta exclusivamente el retorno de ese mismo reward. |
| **51.2 estable en simulación pero mal resultado real** | **Evaluación sin cambio de distribución.** Se evalúa con el mismo `env` y las mismas simplificaciones usadas al entrenar. | `evaluate_policy(Q, env)` llama `env.reset()`/`env.step()` igual que `train`. La baja desviación (`0.9`) mide consistencia dentro de ese simulador, no robustez ante demanda o estados de producción. `new_demand = demand_level` además impide cambios de demanda dentro de cada episodio. |

**Precisión importante:** `epsilon = 0.05` solo explora **acciones**. No puede corregir por sí sola una transición que mantiene fija la demanda ni una distribución de `reset()` que omita estados. Aunque existe `LinearApproximator`, `train` y `evaluate_policy` usan exclusivamente la tabla `Q`, por lo que no hay generalización a tuplas nuevas.


### Entregable 7.2 — Política greedy en estados no visitados


In [2]:
# Auditoría reproducible de la estimación y de la política greedy.
TOTAL_ESTADOS_DECLARADOS = 200
PORCENTAJE_PEDIR_50 = 0.94
ACCIONES_EN_UNIDADES = [0, 10, 20, 30, 40, 50]

# Lectura prevista del dato agregado: si el 94% se calculó sobre los 200,
# 6% (= 12 estados) no elige pedir 50. Todo estado no visitado cae en este
# grupo porque conserva seis ceros y argmax desempata hacia el índice 0.
estados_que_no_eligen_50 = round(
    TOTAL_ESTADOS_DECLARADOS * (1 - PORCENTAJE_PEDIR_50)
)

q_estado_no_visitado = np.zeros(6)
accion_idx = int(np.argmax(q_estado_no_visitado))
accion_unidades = ACCIONES_EN_UNIDADES[accion_idx]
siguiente_estado_critico = transition((0, 1, 'crítico'), accion_unidades)

print(f"Estados fuera del 94%: {estados_que_no_eligen_50}/200")
print("Estimación probable de no visitados: ≈12/200 (6%), no un conteo exacto")
print(f"Q(s) no visitado: {q_estado_no_visitado}")
print(f"argmax -> índice {accion_idx} -> pedir {accion_unidades} unidades")
print(f"Desde (0, 1, 'crítico') se pasa a {siguiente_estado_critico}")


Estados fuera del 94%: 12/200
Estimación probable de no visitados: ≈12/200 (6%), no un conteo exacto
Q(s) no visitado: [0. 0. 0. 0. 0. 0.]
argmax -> índice 0 -> pedir 0 unidades
Desde (0, 1, 'crítico') se pasa a (0, 1, 'crítico')


#### 1. ¿Cuántos estados probablemente no se visitaron?

La estimación más directa que permite el dato agregado es **aproximadamente 12 de 200 estados (6%)**:

- La política reportada pide 50 en 94% de los 200 estados: $0.94(200)=188$.
- Quedan $200-188=12$ estados donde la acción greedy no es 50.
- Un estado no visitado conserva `Q[s] = [0,0,0,0,0,0]` y, por tanto, elige la acción 0; necesariamente queda fuera del 94% que elige 50.

Esta cifra es una **estimación**, no una igualdad demostrable. Formalmente, si el 94% fue calculado sobre los 200 estados, el código permite concluir que hay **como máximo 12** no visitados; algunos de esos 12 también podrían ser estados visitados que aprendieron legítimamente otra acción. La aproximación “12 no visitados” supone que las excepciones a la política dominante se deben principalmente a valores Q que nunca se actualizaron.

El riesgo de cobertura es plausible porque `new_demand = demand_level` impide cambiar de categoría de demanda dentro de un episodio, y `epsilon = 0.05` solo explora acciones, no categorías de estado. Sin embargo, el archivo no incluye `env.reset()`, el horizonte, la tabla Q final ni un conjunto `visited_states`; por eso no es válido fabricar un conteo exacto a partir de `len(Q)`. `Q` también puede contener estados consultados para *bootstrap* o evaluación, y la transición genera coordenadas fuera de la grilla, como días 29, 28, etc.

#### 2. ¿Qué hace la política greedy en un estado no visitado?

`Q = defaultdict(lambda: np.zeros(6))` crea seis ceros al consultar una tupla desconocida. Tanto `train` como `evaluate_policy` usan `np.argmax(Q[state])`. Cuando todos los valores empatan, NumPy devuelve la **primera** posición: índice 0. Con el orden declarado `[0, 10, 20, 30, 40, 50]`, el agente decide **pedir 0 unidades**. Crear la entrada al consultarla no significa que el estado haya sido aprendido.

#### 3. ¿Cómo genera stockouts?

En un estado de inventario bajo y demanda alta o crítica, ordenar cero hace que la demanda supere a `inventory + action`. La transición oculta el faltante al recortar con `max(0, ...)`: por ejemplo, desde `(0, 1, 'crítico')`, cuya demanda diaria es 40, la acción 0 devuelve de nuevo inventario 0. Como el reward no registra unidades de demanda insatisfecha y el código de producción no muestra aprendizaje en línea, la política puede repetir “no pedir” y prolongar el quiebre. Este mecanismo es consistente con los 23 stockouts observados, aunque el código no permite afirmar que explique exactamente los 23.

#### Controles de consistencia

1. El 94% debe tener como denominador los **200 estados** para sostener la estimación de 12. Si fue calculado solo sobre claves visitadas/presentes en `Q`, no informa cuántos estados faltan y el conteo exacto queda indeterminado.
2. La especificación tiene una contradicción adicional: `[0, 10, ..., 100]` contiene 11 valores, no 10; la grilla literal tendría $11 × 5 × 4 = 220$ estados. Aquí se usa 200 porque es el total exigido por el enunciado.
3. Para obtener el número real habría que registrar `visited_states.add(state)` dentro de `train` y comparar ese conjunto con la grilla válida; consultar después un `defaultdict` no es una medición confiable.

**Conclusión:** bajo la lectura prevista de la métrica del 94%, **unos 12 de 200 estados probablemente no se visitaron**. En cualquiera de ellos la política greedy elige **no pedir**, lo que convierte una falta de cobertura del entrenamiento en stockout cuando el estado desconocido tiene poco inventario y demanda elevada.


#### Actualización tras recibir el insumo del Grupo 3 (sesión presencial)

Nuestra estimación de "~12/200 no visitados" arriba asumía que el denominador correcto era 200 estados (el número declarado en el enunciado). El Grupo 3 replicó el entorno original 200 veces (200 semillas × 1000 episodios) y midió directamente **4,371 estados alcanzables** — confirma por otra vía lo que ya habíamos señalado en 7.1: `transition()` no discretiza el inventario a la grilla `[0,10,...,100]`, así que el espacio real de estados es mucho mayor al nominal.

Con ese denominador, la cobertura final de pares `(s,a)` que reporta G3 es **1370.8 / 26226 (≈5.2%)** para el algoritmo original — no ~94% de 200 estados aprendidos, sino una fracción pequeña de un espacio mucho más grande. Esto **refuerza, no contradice**, la conclusión cualitativa de 7.2 (la política greedy elige "pedir 0" por defecto en estados no vistos y eso genera stockouts), pero el problema de cobertura es órdenes de magnitud más grave de lo que el análisis inicial, basado solo en el código y el 94% reportado, permitía estimar. Ajustamos esta cifra en la Pregunta 2 con el detalle completo del bug identificado por G3.


### Entregable 7.3 — Protocolo de evaluación mejorado

El protocolo actual (`evaluate_policy`) falla porque reutiliza el mismo `env` y la misma distribución de estados que `train`: mide consistencia interna del simulador, no transferencia a producción. Proponemos un protocolo de tres capas que se ejecuta **antes** de cualquier despliegue:

**Capa 1 — Métricas de cobertura de entrenamiento (gate previo a evaluar)**
- `estados_visitados / 200` y `pares_(s,a)_visitados / 1200`.
- Condición de aceptación: ≥ 95% de cobertura de estados y ≥ 90% de cobertura de pares `(s,a)`, con cobertura mínima del 80% dentro de cada categoría de demanda (incluyendo `crítico`), no solo en promedio global.
- Si no se cumple, el entrenamiento se rechaza antes de tocar métricas de recompensa — cobertura insuficiente invalida cualquier resultado posterior (ver Entregable 7.2).

**Capa 2 — Evaluación fuera de distribución (estrés dirigido)**
- Conjunto de prueba separado que sobre-muestrea deliberadamente los estados de baja frecuencia: inventario bajo (0–20) combinado con demanda `alto`/`crítico`, y `dias_hasta_vencimiento` bajo (1–7).
- Métrica: tasa de `stockout` simulado (`next_inventory == 0` bajo demanda no satisfecha) y tasa de vencimiento simulado, medidas por separado del reward agregado — un reward promedio saludable puede ocultar colas malas.
- Condición de aceptación: tasa de stockout simulado en el subconjunto de estrés no debe superar en más de 2× la tasa observada en el conjunto de prueba general.

**Capa 3 — Réplica de KPIs de negocio, no solo de reward**
- Traducir el reward a las mismas unidades que reporta producción: stockouts/semana, vencidos/semana, costo de almacenamiento/semana, satisfacción. `evaluate_policy` actual no calcula ninguno de estos directamente.
- Condición de aceptación: proyección de costo de almacenamiento simulado dentro de ±20% del costo objetivo (3200), antes de autorizar el paso a producción.
- Piloto controlado: desplegar la política nueva en un subconjunto pequeño de tiendas/productos durante 1–2 semanas y comparar KPIs reales contra la proyección de la Capa 3 antes del rollout completo.

Este protocolo detecta el gap actual (51.2 en simulación vs. 23 stockouts reales) porque la Capa 1 habría bloqueado el despliegue por cobertura insuficiente, y la Capa 3 habría expuesto que el reward optimizado no corresponde a los KPIs reales de negocio.


### Entregable 7.4 — Dictamen técnico para gerencia

**Para:** Gerencia de Operaciones — Cadena de Farmacias
**De:** Equipo de Consultoría Técnica, Grupo 7 (Evaluación en Producción)
**Asunto:** Causas del gap entre simulación y producción del agente de reabastecimiento

**Resumen ejecutivo.** El agente reporta una recompensa promedio de 51.2 en simulación, pero en producción genera 23 stockouts semanales, 41 unidades vencidas semanales y un costo de almacenamiento de \$8,400 (2.6× el objetivo de \$3,200). Con la información de los Grupos 1, 2, 3, 5 y 6, el diagnóstico se revisa respecto a nuestra lectura inicial: la causa dominante no es solo cobertura de exploración, sino una combinación de **reward mal diseñado (comprobado analíticamente) y una explosión del espacio de estados real** que hace que el algoritmo no converja de forma sana — ninguno de los tres (reward, algoritmo, exploración) resuelve el problema por sí solo, y el MDP corregido resulta ser, según la evidencia cuantitativa recibida, la corrección individual de mayor impacto.

**Causas confirmadas (evidencia en código propio y de los Grupos 2, 3, 5 y 6):**
1. **Reward hacking, estructural y demostrado analíticamente.** `inventory_reward` tiene rango [0,50]; la penalización combinada máxima es -12 — ninguna combinación de penalizaciones compensa inventario alto, así que el óptimo de esa función es pedir el máximo siempre. Confirmado independientemente por el Grupo 2 (sobre 220 estados) y por el Grupo 6 (análisis de rango).
2. **No-convergencia real, no "convergencia sana hacia óptimo local".** La tabla de curvas del sistema original (TD 8.42→5.21, varianza 0.8, política 94% pedir_50) **no es reproducible**: tres grupos independientes (G3, G5, G6), con distintas semillas y configuraciones, midieron un espacio de estados real de 1,673 a 4,371 estados (contra 200 nominales) porque `transition()` no respeta la grilla declarada. La reproducción de G6 (seed=123) da TD error de cola 51.33 (sube, no baja), varianza 127,917, y política dominante `pedir_0` en 80.2% de estados — el signo opuesto al 94% pedir_50 que reporta el enunciado. **Recomendamos a gerencia no confiar en las métricas de entrenamiento del sistema original tal como fueron reportadas** — no son reproducibles de forma consistente entre grupos.
3. **La corrección de mayor impacto individual es el MDP, no el algoritmo ni la exploración.** Según la ablación cuantitativa del Grupo 6 (mismo entorno y semilla, 5 configuraciones comparadas): el MDP corregido de G1, aplicado solo, ya rompe el colapso hacia `pedir_0` (80.2%→48.9%) y da el mejor TD error de toda la tabla (8.42). El algoritmo (G3) y la exploración (G5), aplicados solos en ese mismo pipeline, **no mejoran nada** frente al baseline (G3 casi igual; G5 empeora TD y varianza) — contraintuitivo frente a nuestra hipótesis inicial de que la exploración era la pieza crítica.
4. **Evaluación sin poder de detección.** `evaluate_policy` mide el mismo entorno y reward que `train`; no puede exponer ninguno de los problemas anteriores antes del despliegue (Entregable 7.3).

**Advertencia importante para el plan de corrección:** la ablación de G6 muestra que aplicar **las cuatro correcciones juntas no es estrictamente mejor** que aplicar reward (G2) solo en términos de diversidad de política (74.5% concentrada en `pedir_10` combinado, contra 25.5% con G2 solo). Recomendamos a gerencia un despliegue por fases — MDP + reward primero, medir, y solo agregar algoritmo/exploración si la cobertura en demanda crítica sigue siendo insuficiente — en vez de aplicar todo de una vez asumiendo que la suma es estrictamente mejor.

**Discrepancias metodológicas entre grupos, sin resolver (transparencia hacia gerencia):**
- G3 (entorno sin discretizar, 4,371 estados) y G5 (entorno discretizado, 200 estados) miden cobertura sobre definiciones de entorno distintas — sus porcentajes no son comparables directamente.
- El enunciado original afirma "pedir_50 en 94%"; las reproducciones de G3/G5/G6 apuntan más bien a un colapso hacia "pedir 0". No podemos afirmar cuál cifra es "la correcta" sin que los grupos reconcilien semillas y configuraciones.

**Nota sobre trazabilidad:** el insumo del Grupo 6 (tabla de ablación) se recibió como texto; el notebook fuente (`S10 - Verificacion Empirica Reward Hacking.ipynb`) no fue compartido con nosotros, por lo que no pudimos ejecutarlo ni verificarlo de forma independiente. Lo citamos textual, con esa salvedad explícita.

**Información que aún nos falta para cerrar el diagnóstico:** la fila específica "G1+G2 combinados (sin G3/G5)" en la tabla de ablación de G6 — no está en lo que recibimos, y es la comparación más directa para responder con precisión la Pregunta 1 de integración.


### Insumo recibido — Grupo 1 (MDP corregido)

Entregable 1.3/1.4 del Grupo 1: `mdp_corregido.py` (código completo en la carpeta del proyecto). Cambios sobre el MDP original:

- Estado pasa de 3 a 5 variables: agrega `unidades_en_transito` (pedido de ayer aún en camino — restaura la propiedad de Markov, Entregable 1.1) y `tendencia_demanda` (permite que la demanda cambie dentro del episodio en vez de quedar congelada, Entregable 1.2).
- `transition()` ahora es estocástica (`np.random.choice` sobre `PROB_DEMANDA`/`PROB_TENDENCIA`) y devuelve también `demanda_no_atendida`, cantidad exacta de unidades en falta — dato que la transición original descartaba al recortar con `max(0, ...)`.
- Incluye `estado_a_original()` / `estado_desde_original()` para proyectar entre el formato de 3 y de 5 variables, y `transition_determinista()` para depurar sin ruido aleatorio.

Ejecutamos su `tabla_impacto()` para verificar el código y confirmar las cifras antes de citarlas en las preguntas de integración:


In [3]:
import mdp_corregido as g1

impacto_g1 = g1.tabla_impacto()


Dimension                                Original    Corregido
Niveles de inventario                          10           10
Dias hasta vencimiento                          5            5
Niveles de demanda                              4            4
Unidades en transito (nuevo)                  ---            6
Tendencia de demanda (nuevo)                  ---            3

Total de estados                              200         3600
Pares (estado, accion)                       1200        21600
Entradas de la tabla Q                       1200        21600

Factor de expansion: 18.0


### Insumo recibido — Grupo 2 (recompensa corregida)

Entregable 2.1/2.3/2.4 del Grupo 2 (`Grupo_2_Para_Integracion.ipynb`):

- **2.1 (reward hacking demostrado):** sobre 220 estados, `R_original(pedir 50) - R_original(pedir 0)` es siempre positivo (mínimo 0.5) — confirma matemáticamente lo que ya inferimos en 7.1: pedir el máximo domina en todos los estados bajo el reward original.
- **2.3 (reward corregido, 5 componentes):** `servicio` (satura en 8, no sigue creciendo con más pedido), `faltante` (-2/unidad no atendida), `exceso` (-0.15/unidad sobre 2 días de demanda), `vencimiento` (riesgo gradual: `-0.45 * max(0,7-días)/7 * inventario`, ya no es el escalón binario del original), `pedido` (costo convexo `-0.04q - 0.002q²`, penaliza más pedir 50 que pedir 10). Con esto, la mejor acción ya varía por estado (ej. `(0,1,'crítico')` → pedir 40; `(100,14,'crítico')` → pedir 0), a diferencia del original donde 50 siempre gana.
- **2.4 (simulación comparativa, 1000 episodios × 30 pasos, semilla 3104):** `pedir_50` da recompensa media **-842.29** (inventario promedio 97.98, ventana de vencimiento 18.11/30 pasos, 1500 unidades pedidas); `política_corregida` da **+144.40** (inventario promedio 5.76, ventana de vencimiento 7.48/30, 582.94 unidades pedidas). El reward corregido invierte el signo de la política dominante actual.
- **Dato clave para Pregunta 2 (causa raíz):** en su simulación, `stockout_promedio = 0` **para ambas políticas**, incluyendo `pedir_50`. Es decir: con la transición determinista tal cual está en `S10 - Codigo Examen.py`, ni el reward original ni el corregido generan stockouts por sí solos — la transición nunca dejaría inventario por debajo de la demanda si se pide lo suficiente. Esto es evidencia a favor de que los 23 stockouts de producción **no se explican por el diseño del reward ni de la transición determinista**, sino por algo que este experimento no captura: cobertura de exploración insuficiente (Q-learning tabular con estados nunca visitados, Entregable 7.2) y/o demanda real más variable que el `demand_map` fijo (Entregable 1.1/1.2 del Grupo 1).
- **Handoff explícito de G2 para nosotros:** usar los 41 vencimientos y el costo 2.63× como evidencia compatible con sobreabastecimiento: "no asignar los 23 stockouts únicamente a la recompensa."


### Insumo recibido — Grupo 5 (exploración)

Entregable 5.3/5.4 del Grupo 5 (`parcial.py`, ejecutado y verificado en esta sesión). Su entorno (`PharmacyInventoryEnv`) **discretiza** el inventario a la grilla de 10 niveles (a diferencia del entorno que replicó G3, que no discretiza) — 200 estados nominales, 1200 pares `(s,a)`. Comparan `epsilon-greedy` (ε=0.05, el original) contra `Optimista + UCB` (valor optimista 100 + bono UCB), ambos con 1000 episodios:

| Métrica | ε-greedy (original) | Optimista+UCB (propuesto) |
|---|---|---|
| Pares (s,a) visitados | 770 / 1200 | 1195 / 1200 |
| % cobertura | 64.2% | 99.6% |
| Estados con 0 visitas | 0 / 200 | 0 / 200 |
| Pares no visitados en demanda `crítico` | 150 | 5 |
| Estados con stockout directo (política greedy pide menos de lo que hace falta) | 6 | — |
| De esos, con ≤3/6 acciones probadas ("pobre exploración") | 5 | — |

**Ejecutado por nosotros para verificar:** corre limpio, resultados reproducibles con semilla 42. Los 6 estados con stockout directo son todos de inventario bajo (10-30) y demanda alta/crítica — coherente con nuestra hipótesis de 7.2, pero la cifra real (6 estados, 5 por poca exploración) es mucho menor que la que habíamos estimado con la lógica del 94% (~12/200) porque este entorno (200 estados discretizados) es distinto del que usa G3 para su cifra de 4,371.


In [4]:
import subprocess

resultado_g5 = subprocess.run(
    ['python3', 'exploracion_grupo5.py'],
    capture_output=True, text=True, cwd='.'
)
print(resultado_g5.stdout)


Espacio: 200 estados x 6 acciones = 1200 pares (s,a)

Entrenando epsilon-greedy (epsilon=0.05)
Entrenando Optimista + UCB

METRICA                                  eps-greedy    Opt+UCB
Pares (s,a) visitados                           770       1195
% cobertura                                   64.2%      99.6%
Pares NO visitados                              430          5

Pares NO visitados por nivel de demanda:
  bajo       eps-greedy:   90    Opt+UCB:    0
  medio      eps-greedy:   82    Opt+UCB:    0
  alto       eps-greedy:  108    Opt+UCB:    0
  crítico    eps-greedy:  150    Opt+UCB:    5

Pares NO visitados por accion:
  pedir  0   eps-greedy:    4    Opt+UCB:    0
  pedir 10   eps-greedy:   78    Opt+UCB:    0
  pedir 20   eps-greedy:   88    Opt+UCB:    0
  pedir 30   eps-greedy:   82    Opt+UCB:    1
  pedir 40   eps-greedy:   89    Opt+UCB:    2
  pedir 50   eps-greedy:   89    Opt+UCB:    2

DISTRIBUCION DE VISITAS POR PAR (s,a)

  eps-greedy:
    Pares con 0 visitas:   

### Insumo recibido — Grupo 6 (convergencia)

Entregable 6.1/6.4 del Grupo 6, texto y tabla (fuente: `S10 - Verificacion Empirica Reward Hacking.ipynb`, seed=123, 1000 episodios — **archivo no adjunto a nuestro pull, no lo ejecutamos ni verificamos nosotros mismos**; se cita textual como lo recibimos).

**6.1 — Diagnóstico: ambos (reward hacking + no-convergencia), no "convergencia sana hacia óptimo local".**
- **Reward hacking, estructural (analítico):** `inventory_reward` tiene rango [0,50]; la penalización combinada máxima (`expiry_penalty + order_penalty`) es -12. Ninguna combinación de penalizaciones compensa inventario alto → el óptimo de esa función es pedir el máximo siempre, sin importar demanda ni vencimiento. Es un diseño roto, no un efecto secundario menor.
- **No-convergencia real (empírico, contradice la narrativa original):** su reproducción del baseline dio TD error de cola **51.33 (sube, no baja)**, varianza de reward **127,917** (no el 0.8 que reporta el sistema original), y política dominante real **pedir_0 en 80.2%** de estados (no pedir_50 en 94% como afirma el enunciado). Causa técnica: `transition()` no respeta la grilla de 200 estados (demanda 5/15/25 no es múltiplo de las acciones de a 10) → el estado real visitado explota a **1,673 estados** en su corrida, contra 200 nominales.
- **Cruce explícito con G3:** su hallazgo (TD original 33.8→35.3, no baja de forma consistente) coincide con esta reproducción. Recomiendan retirar la lectura "convergencia sana" basada en la tabla 8.42→5.21 del sistema original — no es reproducible con el código tal cual está.

**6.4 — Proyección/ablación con las 4 correcciones ya recibidas (mismo entorno y semilla):**

| Experimento | Reward | TD error | Varianza | Acción dominante |
|---|---|---|---|---|
| Baseline original | 289.62 | 51.33 | 127,917 | pedir_0 (80.2%) |
| Solo G1 (MDP) | -10.70 | 8.42 (mejor) | 21,185 | pedir_10 (48.9%) |
| Solo G2 (reward) | -627.73 | 35.80 | 39,849 | pedir_10 (25.5%, más repartida) |
| Solo G3 (algoritmo) | 93.03 | 11.09 | 57,028 | pedir_0 (80.7% ≈ baseline) |
| Solo G5 (exploración) | 590.70 | 94.76 (peor) | 226,156 (peor) | pedir_0 (47.8%) |
| **Las 4 combinadas** | -636.60 | 23.14 | 42,612 | pedir_10 (74.5%) |

**Lectura de G6 por corrección:** G1 solo rompe el colapso en `pedir_0` y da el TD más limpio de la tabla — ataca la causa raíz (explosión de estados). G2 solo es el que más diversifica la política por estado (25.5%, mínimo de la tabla) — mejor candidato teórico contra stockout, pero sin G1 no converge limpio (TD 35.8). G3 solo casi no mueve nada frente al baseline. G5 solo empeora todo (peor TD, peor varianza) en su pipeline combinado. Las 4 juntas bajan el TD (23.14) pero la política vuelve a concentrarse (74.5% en `pedir_10`) — mejor que el baseline pero *menos* adaptativa que G2 solo.

**Respuesta directa de G6 a nuestra hipótesis de Pregunta 1:** parcial, con matiz. MDP (G1) solo **ya ataca stockouts**, no solo costo — rompe el `pedir_0` que causa los 23 stockouts/semana. Reward (G2) solo es el que más diferencia política por demanda, pero desestabiliza el TD sin G1. Algoritmo (G3) y exploración (G5) solos no aportan nada por sí mismos en su pipeline — sirven para estabilizar el paquete combinado, no lo sustituyen. Advertencia de G6: combinar las 4 no da lo mejor de cada una — la política combinada (74.5% concentrada) retrocede frente a G2 solo (25.5%); el plan mínimo viable con las 4 correcciones puede seguir sin resolver bien los stockouts en demanda alta/crítica si `pedir_10` no escala con el nivel de demanda.


## Insumos pendientes de otros grupos

> Llenar esta tabla durante la sesión presencial. No borrar las preguntas — son la guía de qué pedirle a cada grupo.

| Grupo | Qué necesitamos | Pregunta concreta a hacerles | Estado |
|---|---|---|---|
| 1 (MDP) | Entregable 1.3 (MDP corregido) y 1.4 (tabla de tamaño de espacio) | ¿Cuántos estados tiene el MDP corregido? ¿La nueva transición de demanda hace que los estados críticos se visiten más seguido de forma natural, o el problema de cobertura persiste igual? | ✅ Recibido (`mdp_corregido.py`) |
| 2 (Recompensa) | Entregable 2.3 (función corregida, 3+ componentes) y 2.4 (tabla comparativa de recompensa acumulada) | ¿Cuál es la magnitud de cada componente? ¿Cuánto reduce la recompensa acumulada de "pedir siempre 50" frente a una política razonable? | ✅ Recibido (`Grupo_2_Para_Integracion.ipynb`) |
| 3 (Algoritmo) | Entregable 3.3 (train corregido) y 3.4 (curvas original vs. corregido) | ¿Cuál era el error exacto en `train`? ¿Qué algoritmo implementaba realmente el código original? ¿Cómo cambia el error TD y la varianza con la corrección? | ✅ Recibido (respuesta directa + `resumen_ablacion.json`, código no adjunto) |
| 4 (Aproximación) | Entregable 4.3 (preprocess_state mejorado) y 4.4 (tabla de MSE) | ¿Qué normalización usaron? ¿Sus características nuevas dependen de la representación de estado actual o de la corregida por el Grupo 1? | ✅ Recibido (respuesta directa, `preprocesamiento_mejorado.py` no adjunto) |
| 5 (Exploración) | Entregable 5.3 (métricas de cobertura) y 5.4 (stockouts atribuibles a no-cobertura) | ¿Cuántos estados y pares (s,a) de los 200/1200 se visitaron realmente en 1000 episodios con ε=0.05? ¿Cuántos de los 23 stockouts atribuyen a estados no visitados? | ✅ Recibido (`parcial.py`, ejecutado y verificado) |
| 6 (Convergencia) | Entregable 6.1 (diagnóstico: óptimo local / reward hacking) y 6.4 (proyección de curvas post-corrección) | ¿Su diagnóstico es reward hacking, óptimo local, o ambos? ¿Cómo proyectan que cambie el error TD y la política dominante si se aplican las correcciones de los Grupos 1, 2 y 3 juntas? | ✅ Recibido (tabla de ablación, texto — **archivo `S10 - Verificacion Empirica Reward Hacking.ipynb` NO adjunto, no ejecutado ni verificado por nosotros**) |

**Regla de registro:** al recibir un insumo, pegarlo textualmente en la celda de la Pregunta de integración correspondiente (no resumir de memoria) y marcar el estado como ✅ aquí.

**Hallazgo transversal de G3, aplica a nuestro propio Entregable 7.2:** su réplica encontró **4,371 estados alcanzables** en el entorno original, no los 200 (ni los 220 literales de la grilla) que usamos como denominador en 7.2. Es consistente con lo que ya habíamos notado en 7.1: `transition()` no discretiza el inventario a la grilla declarada. Esto significa que nuestra estimación de "~12/200 estados no visitados" en 7.2 probablemente **subestima drásticamente** el problema de cobertura real — ver la nota agregada al final de 7.2 y la respuesta actualizada de Pregunta 2.

**Contradicción metodológica G3 vs. G5 (importante, sin resolver):** el entorno de G3 replica `transition()` del `.py` original **sin discretizar** el inventario (de ahí sus 4,371 estados alcanzables). El entorno de G5 (`parcial.py`) **sí discretiza** el inventario con `_discretize_inventory()` a la grilla de 10 niveles — su `PharmacyInventoryEnv` tiene exactamente 200 estados nominales y ningún estado con 0 visitas tras 1000 episodios. Son dos ambientes distintos probando la misma pregunta de cobertura; sus porcentajes de cobertura **no son comparables directamente** entre sí (5.2% de G3 sobre 26,226 pares vs. 64.2% de G5 sobre 1,200 pares).

**Tercer dato de tamaño de estado — G6 (reconcilia, no contradice):** su reproducción del baseline (seed=123, 1000 episodios) visitó **1,673 estados** de forma empírica en una sola corrida — es un subconjunto razonable de los 4,371 "alcanzables" que G3 midió por enumeración, no una tercera cifra contradictoria. Confirma otra vez, por una cuarta vía independiente, que `transition()` no respeta la grilla de 200 estados declarada.

**Nueva discrepancia — G5 vs. G6 sobre el efecto de la exploración sola:** G5 reportó que su estrategia Optimista+UCB, aislada, sube la cobertura de 64.2% a 99.6% (mejora clara). G6, en su tabla de ablación "Solo G5" dentro de su propio pipeline combinado, reporta que la exploración sola **empeora** TD error (94.76, el peor de la tabla) y varianza (226156, la peor). No es necesariamente una contradicción — probablemente miden cosas distintas (cobertura vs. TD/varianza de reward) sobre setups distintos (aislado vs. combinado con el resto del pipeline de G6) — pero hay que preguntarles a ambos grupos cómo corrieron "G5 solo" antes de usar esa fila de la tabla de G6 para decidir prioridades.


## Preguntas de integración

### Pregunta 1 — Diagnóstico sistémico

**Insumos usados:** MDP corregido (Grupo 1, ✅ recibido) · función de recompensa corregida (Grupo 2, ✅ recibido) · proyección de curvas (Grupo 6, ✅ recibido — tabla de ablación, archivo fuente no verificado por nosotros).

**Respuesta con evidencia cuantitativa directa de G6 (ablación real, no especulación nuestra):** aplicar únicamente MDP (G1) y reward (G2), sin algoritmo ni exploración, **sí mejora la política** — pero con un matiz importante que revisa nuestra hipótesis original.

- **MDP (G1) solo** ya rompe el colapso hacia `pedir_0` (80.2%→48.9% de estados) y da el TD error más bajo de toda la tabla de G6 (8.42, mejor que cualquier otra combinación, incluida las 4 juntas). Esto **ataca directamente los stockouts**, no solo el costo de almacenamiento como pensábamos — la causa (estados no visitados eligiendo "no pedir") se resuelve en buena parte con solo tener un MDP que restaura la propiedad de Markov y no explota el espacio de estados de forma descontrolada.
- **Reward (G2) solo** es, según la tabla de G6, el que **más diversifica la política por nivel de demanda** (`pedir_10` en solo 25.5% de estados — el mínimo de toda la tabla, es decir la política menos concentrada, más sensible al estado real). Es el mejor candidato individual contra el problema de sobreabastecimiento/stockout combinado, pero **no converge limpio sin G1** (TD sube a 35.80).
- **Juntos (G1+G2, sin G3 ni G5):** no tenemos la fila exacta de esa combinación en la tabla de G6 (solo da "solos" y "las 4 juntas"), así que no podemos afirmar el número exacto — es la primera pieza de información que aún nos falta pedirle a G6 para esta pregunta específica.

**Revisión de nuestra hipótesis original:** habíamos concluido que MDP+reward solos "mejoran costo pero no stockouts, hace falta algoritmo/exploración". La evidencia de G6 la **matiza, no la confirma tal cual**: el MDP de G1 por sí solo ya reduce fuertemente el colapso en `pedir_0` (la causa directa de stockout que identificamos en 7.2), y el algoritmo (G3) y la exploración (G5), aplicados solos en el pipeline de G6, **no aportan nada por sí mismos** (G3 ≈ baseline; G5 empeora TD y varianza). Es decir: la corrección de más impacto individual no es la que nuestra hipótesis original priorizaba (algoritmo/exploración), sino el MDP. Dicho esto, G6 advierte que las 4 correcciones combinadas retroceden en diversidad de política (74.5% concentrada en `pedir_10`, peor que G2 solo con 25.5%) — el paquete completo estabiliza el entrenamiento (TD 23.14) pero puede seguir sin resolver bien los estados de demanda alta/crítica si `pedir_10` no escala con la demanda. Esto reabre la pregunta de si el orden de aplicación (y no solo el conjunto de correcciones) determina el resultado — ver Pregunta 3.

### Pregunta 2 — Causa raíz

**Insumos usados:** algoritmo corregido (Grupo 3, ✅ recibido) · métricas de cobertura (Grupo 5, ✅ recibido) · análisis de gap (Grupo 7, propio: Entregables 7.1/7.2).

**Grupo 2 (referencia cruzada):** en su simulación con política óptima computada directamente (sin aprendizaje tabular), `stockout_promedio = 0` bajo cualquier política — apuntaba a que la causa está en el *proceso de aprendizaje*, no en el reward ni en la transición.

**Grupo 3 — el bug real (no es SARSA accidental):** la fórmula de actualización es Q-learning off-policy correcto matemáticamente. El bug real es que `np.argmax` **no desempata al azar**: con `Q` inicializado en ceros, todo estado no visitado empata en sus 6 acciones y la política greedy elige siempre el índice más bajo — **acción 0 (no pedir)**. Sobre su réplica del entorno **sin discretizar** (4,371 estados alcanzables), la cobertura final de pares `(s,a)` es solo **5.23% de 26,226**.

**Grupo 5 — cobertura medida directamente sobre el entorno discretizado (200 estados, 1200 pares):** ε-greedy (ε=0.05) cubre 64.2% de los pares tras 1000 episodios; el déficit se concentra en demanda `crítico` (150/1200 pares sin visitar). Esto produce **6 estados con stockout directo** bajo la política greedy, **5 de los cuales** tienen 3 o menos de las 6 acciones probadas.

**Grupo 6 — confirma con una cuarta medición independiente que el espacio real explota más allá de la grilla declarada:** su reproducción del baseline visitó **1,673 estados** en una sola corrida (subconjunto razonable de los 4,371 "alcanzables" de G3), y encontró que la política dominante real del sistema original es **`pedir_0` en 80.2% de estados** — no `pedir_50` en 94% como afirma el enunciado del examen. Esto es una corrección directa a un dato que habíamos tomado como dado desde 7.1/7.2: la "política dominante" reportada en el sistema original **no es reproducible tal cual**, y en la reproducción de G6 va en la dirección opuesta (colapso hacia "no pedir", no hacia "pedir el máximo").

**Contradicción/matiz a resolver (no la escondemos):** el enunciado original afirma "pedir_50 en 94% de estados"; nuestro propio análisis (7.1) tomó ese dato como evidencia de reward hacking hacia sobreabastecimiento. La reproducción de G6 (pedir_0 en 80.2%) y de G3/G5 (colapso hacia baja cobertura → default a acción de índice más bajo) apuntan más bien a que el comportamiento real y reproducible del sistema es **subabastecimiento por defecto**, no sobreabastecimiento. Ambos pueden ser ciertos en simultáneo si el 94%/pedir_50 del enunciado viene de una corrida o semilla distinta a las que replicaron G3/G5/G6 — pero no podemos afirmar cuál es "la" cifra correcta sin que los grupos reconcilien sus semillas y configuraciones. Para el dictamen (7.4), tratamos ambos síntomas (sobreabastecimiento *y* posible colapso a "no pedir") como plausibles y dependientes de la corrida específica, no como una cifra única y consistente.

**Conclusión de causa raíz:** el problema **no es el reward ni el MDP en sí** (confirmado independientemente por G2 y por la ausencia de stockouts bajo política óptima) — es la combinación de (a) el bug de desempate en `argmax`, que sesga sistemáticamente hacia "no pedir" en estados poco visitados, (b) una tasa de exploración fija (`epsilon=0.05`) que no compensa esa falta de cobertura en demanda crítica, y (c) un espacio de estados real varias veces más grande que el declarado (200 nominal vs. 1,673–4,371 medidos por tres grupos distintos de forma independiente), que garantiza que la cobertura sea estructuralmente insuficiente con el presupuesto de episodios actual.

### Pregunta 3 — Plan de corrección mínimo viable

**Insumos usados:** resultados de todos los grupos, incluida la ablación cuantitativa de G6.

| Prioridad | Componente que modifica | Grupo fuente | Métrica que mejora | Mejora esperada (medida) | Justificación |
|---|---|---|---|---|---|
| 1 | MDP (restaurar Markov, no explotar espacio de estados sin control) | Grupo 1 | TD error y colapso de política hacia `pedir_0` | TD 51.33→8.42 (el mejor de toda la tabla de G6); política dominante 80.2%→48.9% | Según la ablación de G6, es la corrección individual de mayor impacto — ataca la causa raíz de por qué el agente converge mal, más que el algoritmo o la exploración por sí solos |
| 2 | Recompensa | Grupo 2 | Diversidad de política por nivel de demanda; costo de almacenamiento | Política dominante cae a 25.5% (la más repartida de la tabla de G6); inventario promedio 97.98→5.76 unidades (G2) | Es la corrección que más alinea la acción con el estado real, pero G6 muestra que no converge limpio sin G1 (TD 35.8) — debe aplicarse junto con G1, no antes |
| 3 | Algoritmo + exploración (desempate aleatorio, decaimiento α/ε, Optimista+UCB) | Grupos 3 y 5 | Estabilidad del entrenamiento combinado; cobertura en demanda crítica | Según G6, ninguna de las dos aporta por sí sola en su pipeline (G3 ≈ baseline, G5 empeora TD/varianza aislada), pero G5 sube cobertura de 64.2%→99.6% en su propio experimento aislado | Necesarias para estabilizar el paquete completo (G6: TD combinado 23.14) y para que la cobertura en demanda crítica no dependa solo de que G1+G2 "por suerte" visiten esos estados |

**Advertencia explícita de G6 que cambia la recomendación final:** las 4 correcciones juntas no dan lo mejor de cada una — la política combinada (74.5% concentrada en `pedir_10`) es *menos* adaptativa que G2 solo (25.5%). Recomendación: **aplicar G1+G2 primero y medir**; solo agregar G3/G5 si la cobertura en estados de demanda alta/crítica sigue siendo insuficiente después de esas dos, en vez de aplicar las 4 de una vez asumiendo que la suma es estrictamente mejor. Esto es ejecutable en dos semanas: G1 y G2 son reemplazos de función ya escritos y probados por sus respectivos grupos; G3/G5 quedan como fase de estabilización condicional, no como parte obligatoria del primer despliegue.

### Pregunta 4 — Compatibilidad de correcciones

**Insumos usados:** MDP corregido (Grupo 1, ✅ recibido) · preprocesamiento mejorado (Grupo 4, ✅ recibido) · algoritmo corregido (Grupo 3, ✅ recibido).

**Grupo 1 — riesgo de incompatibilidad confirmado en el código:** `mdp_corregido.py` cambia el estado de 3 a 5 variables. El `preprocess_state` original y el `LinearApproximator(n_features=3, ...)` del Grupo 4 están escritos para 3 variables.

**Grupo 4 — respuesta directa:** trabajan enteramente sobre el estado original de 3 variables, expandido a 12 features (normalización correcta, demanda como one-hot). No usan `unidades_en_transito` ni `tendencia_demanda`. Confirman: si se adopta el MDP de 5 variables de G1, necesitan que el estado llegue ya reducido a `(inventory, days_to_expiry, demand_level)`.

**Grupo 3 — respuesta directa:** mecánicamente, `Q = defaultdict(lambda: np.zeros(6))` no tiene problema con tuplas de 5 elementos en vez de 3. El problema real es de **calibración**: su esquema de decaimiento de `alpha`/`epsilon` ya está calibrado contra su propio hallazgo de 4,371 estados alcanzables del MDP *original*. Si el espacio crece más con el MDP de G1, la misma calibración cubre una fracción todavía menor con el mismo presupuesto de episodios.

**Conclusión de compatibilidad — las tres correcciones son técnicamente compatibles, pero no son sinérgicas sin coordinación adicional:**
1. **G1 ↔ G4:** compatible vía `estado_a_original()` (adaptador que ya escribió G1), pero G4 pierde la señal de las 2 variables nuevas — no hay error, hay pérdida de información.
2. **G1 ↔ G3:** compatible en el mecanismo, pero **incompatible en calibración**: aplicar el MDP de G1 sin que G3 recalibre `alpha`/`epsilon` contra el nuevo tamaño de espacio empeora la cobertura que ya es la causa raíz principal (Pregunta 2).
3. **G3 ↔ G5:** ambos corrigen exploración/algoritmo pero sobre **definiciones distintas del entorno** (G3 sin discretizar, G5 discretizado a 200 estados) — antes de combinar sus dos correcciones en un solo pipeline, hay que decidir sobre cuál de los dos entornos se entrena en producción.
4. **G3/G5 ↔ G6:** la ablación de G6 muestra que G3 y G5, aplicados solos dentro de su pipeline, no mejoran (o empeoran) las métricas frente al baseline — mientras que G5, en su propio experimento aislado, sí mejora cobertura sustancialmente. Confirma que el efecto de cada corrección depende fuertemente de con qué otras correcciones y sobre qué entorno se combine — no se puede asumir que los resultados aislados de cada grupo se trasladan igual a un pipeline combinado.
5. **Orden recomendado (revisado con la evidencia de G6):** primero MDP (G1), luego reward (G2) — es la combinación de mayor impacto individual y evita el problema de "las 4 juntas son peor que G2 solo" señalado en Pregunta 3 — y solo agregar algoritmo (G3) y exploración (G5) si, tras medir G1+G2, la cobertura en demanda crítica sigue insuficiente, recalibrando `alpha`/`epsilon` contra el nuevo tamaño de espacio antes de integrarlos.


## Reflexión grupal

> **[PENDIENTE — completar después de la sesión presencial]**
> Media página respondiendo: ¿qué cambió en nuestro diagnóstico inicial (Entregables 7.1/7.2, basado solo en el código y los agregados de producción) después de ver los resultados concretos de los otros grupos? Señalar específicamente qué hipótesis se confirmó, cuál se descartó o se matizó, y con el resultado de qué grupo.
